## 1. Ingestion
Documents containing construction site visit reports are loaded, normalized and chunked. Then these chunks are saved to a JSON file.

In [1]:
# Install python-docx library
%pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 6.3 MB/s eta 0:00:00


In [2]:
# Import libraries for document processing and text handling
import json
import re
import uuid
from pathlib import Path

from docx import Document

In [3]:
# Functions for text normalization and chunking

# Normalization function
def normalize_text(text):
    # Replace multiple whitespace characters (spaces, tabs, newlines) with a single space
    cleaned_text = re.sub(r'\s+', ' ', text)

    # Strip leading and trailing whitespace
    cleaned_text = cleaned_text.strip()

    return cleaned_text

# Returns True if chunk should be discarded, False otherwise
def is_bad_chunk(chunk):
    words = len(chunk.split())

    if words < 40:
        return True

    # Minimum alphabetic character ratio (filters noisy/non-text chunks)
    alpha_ratio = sum(c.isalpha() for c in chunk) / max(len(chunk), 1)
    if alpha_ratio < 0.65:
        return True

    return False

# Chunking function (character based)
def chunk_text_char_based(text, chunk_size=1000, overlap=200):
    chunks = []
    if not text:
        return chunks
    start = 0
    text_length = len(text)

    while start < text_length:

        # Tentative end of the chunk
        end = min(start + chunk_size, text_length)

        # Adjust end to last whitespace to avoid splitting words
        if end < text_length:
            space_pos = text.rfind(" ", start, end)
            if space_pos != -1 and space_pos > start:
               end = space_pos

        chunk = text[start:end].strip()

        # Add chunk to the set of chunks
        if chunk and not is_bad_chunk(chunk):
            chunks.append(chunk)

        # Next window win overlap
        next_start = end - overlap

        # Avoid going backwards
        if next_start <= start:
            start = end
        else:
            # Move forward if overlap lands inside a word
            while next_start < text_length and text[next_start] != " ":
                next_start += 1

            # Skip whitespace
            while next_start < text_length and text[next_start] == " ":
                next_start += 1

            start = next_start

    return chunks


In [4]:
# Extract chunks from documents and store them in records
docs_dir = "./docs"
json_file_name = "chunk_info_list.json"

# docs_paths = Path(docs_dir).glob('*.docx')
docs_paths = list(Path(docs_dir).glob("*.docx")) + list(Path(docs_dir).glob("*.doc"))

chunk_info_list = []

for doc_path in docs_paths:
    # Use filename (without extension) as document identifier
    doc_id = doc_path.stem
    doc = Document(doc_path)

    texts = []
    # Normal paragraphs
    for p in doc.paragraphs:
        text = normalize_text(p.text)
        if not text:
            continue
        texts.append(text)

    for table in doc.tables:
        for row in table.rows:
            row_texts = []
            seen = set()
            for cell in row.cells:
                text = normalize_text(cell.text)
                if not text:
                    continue
                # evita duplicados reales dentro de la fila (no estructurales)
                if text in seen:
                    continue
                seen.add(text)
                row_texts.append(text)

            if row_texts:
                texts.append(" || ".join(row_texts))

    # Reconstruct text
    doc_text = "\n".join(texts)

    # Split text into chunks
    doc_chunks = chunk_text_char_based(doc_text, chunk_size=1000, overlap=200)

    # Collect document chunks and assign each an RFC 9562 UUID
    for chunk in doc_chunks:

        # Filter out bad chunks
        if is_bad_chunk(chunk):
            continue

        chunk_info_list.append({
            "doc_id": doc_id,
            "chunk_id": str(uuid.uuid4()),
            "chunk_text": chunk
        })

# Save document chunks to a JSON file
with open(json_file_name, "w", encoding="utf-8") as f:
    json.dump(chunk_info_list, f, indent=4, ensure_ascii=False)

## Indexing
Embeddings are generated from text chunks using the [BSC-LT/MrBert-es](https://huggingface.co/BSC-LT/MrBERT-es) model. These embeddings are then stored in a [Chroma vector database](https://www.trychroma.com/products/chromadb).

In [5]:
# Install Hugging Face transformers library
%pip install -U transformers

# Install chromadb vector database library
%pip install -U chromadb

# Install rank_bm25 library
%pip install -U rank_bm25


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 93.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
# Import PyTorch
import torch

# import chromadb to store (embedding, chunk pairs)
import chromadb

# import  Hugging Face AutoModel/AutoTokenizer classes for loading pretrained models and tokenizers by name
from transformers import AutoTokenizer, AutoModel

# import BM25Okapi class from rank_bm25 for computing BM25 document scores
from rank_bm25 import BM25Okapi

In [7]:
# Model used for embedding generation: MrBERT-es by BSC-LT
model_name = "BSC-LT/MrBERT-es"

# Load the model and tokenizer from Hugging Face Hub
model = AutoModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.34k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/601M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: BSC-LT/MrBERT-es
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/6.83M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/751 [00:00<?, ?B/s]

In [8]:
# Functions for Indexing
def get_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[:, 0, :].squeeze().tolist()

def tokenize(text):
    return re.findall(r"\b[a-zA-ZáéíóúñÁÉÍÓÚÑ]+\b", text.lower())

In [9]:
# Obtain embeddings and store them in chunk records
chunk_records = []
chunk_text_map = {}

for i, chunk_info in enumerate(chunk_info_list):
    embedding = get_embedding(chunk_info["chunk_text"])

    chunk_records.append({
        "doc_id": chunk_info["doc_id"],
        "chunk_id": chunk_info["chunk_id"],
        "chunk_text": chunk_info["chunk_text"],
        "embedding": embedding
    })

    chunk_text_map[chunk_info["chunk_id"]] = chunk_info["chunk_text"]

# Obtain keyword corpus for lexical search
keyword_corpus = [
    tokenize(chunk_record["chunk_text"])
    for chunk_record in chunk_records
]

# Save chunk records to a JSON file
json_chunk_records_file_name = "chunk_records.json"
with open(json_chunk_records_file_name, "w", encoding="utf-8") as f:
    json.dump(chunk_records, f, indent=4, ensure_ascii=False)

# Save keyword corpus to a JSON file
json_keyword_corpus_file_name = "keyword_corpus.json"
with open(json_keyword_corpus_file_name, "w", encoding="utf-8") as f:
    json.dump(keyword_corpus, f, indent=4, ensure_ascii=False)

In [10]:
# Persist chunk embeddings and metadata in a Chroma collection for RAG retrieval
collection_name = "construction_site_visit_reports"
database_path = "./chroma_db"
client = chromadb.PersistentClient(path=database_path)

# Delete collection if it already exists (to avoid duplicate insertions on re-runs)
try:
    client.delete_collection(collection_name)
except Exception:
    pass

# Create a collection using cosine similarity (HNSW index under the hood)
collection = client.create_collection(
    name=collection_name,
    configuration={
        "hnsw": {
            "space": "cosine"
        }
    }
)

# Add documents (chunks), their embeddings, and metadata to the collection
collection.add(
    ids=[str(chunk_record["chunk_id"]) for chunk_record in chunk_records],
    documents=[chunk_record["chunk_text"] for chunk_record in chunk_records],
    embeddings=[chunk_record["embedding"] for chunk_record in chunk_records],
    metadatas=[{"doc_id": chunk_record["doc_id"],
                "chunk_id": chunk_record["chunk_id"]}  for chunk_record in chunk_records]
)

# Build BM25 index from the keyword_corpus
bm25 = BM25Okapi(keyword_corpus)

print(f"Stored {collection.count()} records in ChromaDB.")
print(f"Stored {len(keyword_corpus)} records in bm25.")


Stored 305 records in ChromaDB.
Stored 305 records in bm25.


## 3. Retrieval

In [11]:
# Functions for Retrieval

# Reciprocal rank fusion (RRF)
def rrf(rank, k=60):
    return 1 / (rank + k)


# Hybrid retrieval: fuse lexical (BM25) and semantic (Chroma) rankings using RRF
def hybrid_search(query_text,
                  k=5,
                  retrieval_k=100,
                  semantic=True,
                  lexical=True):

    # Fusion (Reciprocal Rank Fusion)
    scores = {}

    # Semantic (Chroma) search
    if semantic:
        # Semantic (Chroma) search
        query_embedding = get_embedding(query_text)
        chroma_results = collection.query(
            query_embeddings=[query_embedding],
            n_results=retrieval_k,
            include=["documents", "metadatas", "distances"]
        )

        for rank, chunk_id in enumerate(chroma_results["ids"][0]):
            scores[chunk_id] = scores.get(chunk_id, 0) + rrf(rank)

    # Lexical (BM25) search
    if lexical:
        tokenized_query = tokenize(query_text)
        bm25_scores = bm25.get_scores(tokenized_query)
        top_bm25_idx = sorted(
            range(len(bm25_scores)),
            key=lambda i: bm25_scores[i],
            reverse=True
        )[:retrieval_k]
        for rank, i in enumerate(top_bm25_idx):
           chunk_id = chunk_records[i]["chunk_id"]
           scores[chunk_id] = scores.get(chunk_id, 0) + rrf(rank)

    # Final ranking
    ranked_chunk_ids = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    # Reconstruct chunk texts
    final_chunks = [
        chunk_text_map[chunk_id]
        for chunk_id, score in ranked_chunk_ids
    ]

    return final_chunks[:k]

In [16]:
# Query text
query_text = "instalaciones del ascensor y cubiertas" # @param {type:"string"}

# Retrive
results = hybrid_search(query_text, k=10)
for i, doc in enumerate(results):
    print(f"\nRank {i+1}")
    print(doc)



Rank 1
y demás áreas de obras es la siguiente: Finalización de la ejecución del foso de hormigón armado del ascensor del edículo. Ejecución de los muros de hormigón armado de la rampa. Por efecto de las lluvias en esta semana no se ha podido ejecutar el último tramo de batache. Continuación con el hormigonado del prisma de las canalizaciones de RENFE y ADIF enterrado bajo el andén. Instalación de la red enterrada de baldeo. Construcción de arquetas de registro ADIF y RENFE. Finalización de la ejecución de murete de bloque de hormigón para el recrecido del borde de andén e inicio de su revoco con mortero. Hormigonado del foso y muros del ascensor del edículo. Inicio de la solera de andén desde el lado Mollet. Replanteo de la pieza de borde de andén y de pavimentos podotáctiles.
2. || PLANIFICACIÓN DE OBRA:
2.1. || Trabajos que se están ejecutando en la semana del: Finalización de la ejecución del foso de hormigón armado del ascensor del edículo. Ejecución de los muros de hormigón armad